In [ ]:
import openai

client = openai.OpenAI()

questions = [
    "지금 인기 있는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘"
]

for question in questions:
    prompt = """
나에겐 아래 함수들이 있어.

get_popular_movies() - /movies에서 인기 영화를 가져옵니다.
get_movie_details(id) - /movies/:id에서 영화 정보를 가져옵니다.
get_movie_credits(id) - /movies/:id/credits에서 출연진 및 제작진을 가져옵니다.

규칙:
- 질문에 가장 알맞은 함수 하나만 골라야 해.
- movie ID가 있으면 그 숫자를 그대로 사용해야 해.
- 다른 설명은 하지 말고 함수 호출 형태로만 답해야 해.

출력 형식 예시:
get_popular_movies()
get_movie_details(550)
get_movie_credits(550)

질문:
""" + question

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    answer = response.choices[0].message.content.strip()

    print("질문:", question)
    print("모델 답:", answer)
    print()

In [ ]:
import openai

client = openai.OpenAI() 

messages = []

def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )
    message = response.choices[0].message.content
    messages.append({"role": "assistant", "content": message})

    print(f"AI: {message}")

while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()
        

# 과제 1 - Movie Agent 만들기

## 목표
완전한 에이전트 루프를 갖춘 Movie Agent를 구축하세요!

## 에이전트가 수행해야 할 작업
1. 영화에 대한 사용자 질문을 받습니다.
2. 어떤 도구를 호출할지 결정합니다 (필요한 경우).
3. 실제로 API를 호출하여 LLM에게 실제 결과를 반환합니다.
4. LLM이 최종 답변을 줄 때까지 루프를 계속합니다.

## 사용할 API
- 기본 URL: `https://nomad-movies.nomadcoders.workers.dev`
- `/movies` - 인기 영화 목록
- `/movies/:id` - ID로 영화 상세 정보 조회
- `/movies/:id/credits` - 영화의 출연진 및 제작진 조회
- `/movies/:id/similar` - 유사한 영화 조회

## 실제 API를 호출하는 최소 3개의 도구를 갖춘 에이전트를 만드세요:
- `get_popular_movies()` → `/movies`
- `get_movie_details(id)` → `/movies/:id`
- `get_similar_movies(id)` → `/movies/:id/similar`

## 에이전트가 갖춰야 할 조건:
- (수동 프롬프팅이 아닌) OpenAI tools 파라미터를 사용하세요.
- 응답에서 tool_calls를 처리하세요.
- 실제 API를 호출하고 결과를 다시 전달해야 합니다.
- 메모리를 활용한 멀티턴 대화를 지원해야 합니다.



In [2]:
# =============================================================
#  과제 1 - Movie Agent (니꼬쌤 my-first-agent 소스 참고)
#  완전한 에이전트 루프를 갖춘 영화 추천 에이전트
# =============================================================
#
# [ 전체 순서도 ]
#
#   사용자: "요즘 인기 영화 뭐 있어?"
#        │
#        ▼
#   messages에 사용자 질문 저장
#        │
#        ▼
#   call_ai() → AI한테 messages + TOOLS 넘김
#        │
#        ▼
#   AI 판단: "get_popular_movies() 써야겠다!"
#        │
#        ▼
#   process_ai_response()
#        ├─ AI가 함수 골랐음 → 기록 저장
#        ├─ 진짜 함수 실행 (실제 API 호출!)
#        ├─ 결과를 messages에 저장
#        └─ call_ai() 다시 호출!
#        │
#        ▼
#   AI가 API 결과를 보고 자연스럽게 답변
#        │
#        ▼
#   "현재 인기 영화는 ○○, ○○ 입니다~"
#
# =============================================================

import openai
import json
import requests  # 실제 HTTP API 호출을 위한 라이브러리

client = openai.OpenAI()

# ★ 대화 기록 저장소 (AI의 "기억")
# 이 리스트를 새로 초기화해서 위쪽 셀의 messages와 분리
#
# 맨 처음에 "시스템 프롬프트"를 넣어둠
# → AI한테 "넌 이런 캐릭터고, 이런 식으로 답해야 해" 라고 미리 알려주는 것
# → 사용자 입력 전에 세팅되는 AI의 "성격/규칙"
messages = [
    {
        "role": "system",
        "content": """너는 영화 추천 에이전트야.

[응답 규칙 - 매우 중요!]
- 답변은 최대한 간결하게 해.
- 영화 이름, ID, 줄거리 한 줄 정도만 알려줘.
- 마크다운 링크([제목](url)) 절대 사용하지 마.
- 포스터 이미지 URL, 백드롭 이미지 URL 출력하지 마.
- 평점은 필요할 때만 간단히 표시해.

[출력 예시]
User: 지금 인기 있는 영화 알려줘
Agent: 현재 인기 영화 목록입니다:
1. 듄: 파트 2 (ID: 693134)
2. 오펜하이머 (ID: 872585)
3. 바비 (ID: 346698)
...

User: 듄에 대해 더 알려줘
Agent: 듄: 파트 2는 드니 빌뇌브 감독의 2024년 SF 대작으로, 폴 아트레이데스의 복수를 그립니다.

User: 비슷한 영화 추천해 줄래?
Agent: 듄을 좋아하셨다면 이런 영화를 추천드립니다: 블레이드 러너 2049, 어라이벌, 인터스텔라.
""",
    }
]

# API 기본 URL
BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"


# =============================================================
# [1] 팔다리 만들기 - 실제 API를 호출하는 함수들
# =============================================================

def get_popular_movies():
    """인기 영화 목록을 가져오는 함수
    → GET /movies

    API는 영화 20개를 돌려주고, 각 영화에 14개 필드가 있음.
    너무 많으면 AI가 헷갈리니까 핵심 필드만 골라서 돌려주자!
    """
    response = requests.get(f"{BASE_URL}/movies")
    data = response.json()

    # 핵심 필드만 골라서 간단하게 정리
    # (id, title, overview, 평점, 개봉일)
    simplified = [
        {
            "id": m["id"],                      # ★ 영화 ID (상세 조회에 필요!)
            "title": m["title"],                 # 제목
            "overview": m["overview"][:100],     # 줄거리 (100자까지만)
            "vote_average": m["vote_average"],   # 평점
            "release_date": m["release_date"],   # 개봉일
        }
        for m in data
    ]

    # 화면에도 ID와 제목을 바로 보이게 출력 (사람이 확인하기 편하게)
    print("📋 인기 영화 목록 (ID / 제목):")
    for m in simplified:
        print(f"   - ID {m['id']}: {m['title']}")

    return json.dumps(simplified, ensure_ascii=False)


def get_movie_details(id):
    """특정 영화의 상세 정보를 가져오는 함수
    → GET /movies/:id

    id: 영화 ID (예: 693134)
    """
    response = requests.get(f"{BASE_URL}/movies/{id}")
    data = response.json()
    return json.dumps(data, ensure_ascii=False)


def get_similar_movies(id):
    """특정 영화와 비슷한 영화 목록을 가져오는 함수
    → GET /movies/:id/similar

    id: 기준이 되는 영화 ID
    """
    response = requests.get(f"{BASE_URL}/movies/{id}/similar")
    data = response.json()
    return json.dumps(data, ensure_ascii=False)


# ★ 함수 이름표 → 진짜 함수 연결 사전
# AI가 "get_popular_movies"라는 문자열을 돌려주면
# 이 사전에서 진짜 함수를 찾아서 실행함
FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies,
}


# =============================================================
# [2] AI한테 줄 함수 설명서 (TOOLS)
# =============================================================
# AI는 코드를 볼 수 없음! 이 설명서만 보고 어떤 함수를 쓸지 결정함
# 그래서 description을 꼼꼼하게 잘 써줘야 함

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "현재 인기 있는 영화 목록을 가져옵니다. 사용자가 '인기 영화', '요즘 뭐 봐', '추천 영화' 같은 질문을 하면 사용하세요.",
            "parameters": {
                "type": "object",
                "properties": {},  # 파라미터 없음
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "특정 영화의 상세 정보(줄거리, 감독, 개봉일 등)를 가져옵니다. 영화 ID가 필요합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "상세 정보를 가져올 영화의 ID (숫자)",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "특정 영화와 비슷한 영화 목록을 가져옵니다. 사용자가 '비슷한 영화', '이런 영화 더 추천해줘' 같은 요청을 하면 사용하세요.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "기준이 되는 영화의 ID (숫자)",
                    }
                },
                "required": ["id"],
            },
        },
    },
]


# =============================================================
# [3] 뼈대 조립 - 핵심 로직
# =============================================================

from openai.types.chat import ChatCompletionMessage


def process_ai_response(message: ChatCompletionMessage):
    """AI의 응답을 처리하는 함수
    - AI가 함수 쓰고 싶어하면 → 함수 실행하고 결과를 다시 AI한테
    - AI가 그냥 답변했으면 → 출력하고 끝
    """

    # ─── 경우 A: AI가 함수 호출을 원하는 경우 ───
    if message.tool_calls:

        # [A-1] AI가 "이 함수 쓸래"라고 한 기록을 messages에 저장
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        # [A-2] AI가 고른 함수들을 하나씩 실행
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"🔧 함수 호출: {function_name}({arguments})")

            # [A-3] JSON 문자열 → 파이썬 딕셔너리 변환
            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            # [A-4] 이름으로 진짜 함수 찾아서 실행!
            function_to_run = FUNCTION_MAP.get(function_name)
            if function_to_run is None:
                result = f"Error: '{function_name}' 함수를 찾을 수 없습니다."
            else:
                try:
                    # **arguments: 딕셔너리를 풀어서 넘기는 문법
                    # {"id": 550} → id=550
                    result = function_to_run(**arguments)
                except Exception as e:
                    result = f"Error: {str(e)}"

            print(f"✅ 결과 받음 (앞부분): {str(result)[:120]}...")

            # [A-5] 함수 실행 결과를 messages에 저장
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": str(result),
                }
            )

        # [A-6] ★ 다시 AI한테 물어보기!
        # "함수 결과 줬으니 이제 사용자한테 답변해줘"
        # 이 재귀 호출 덕분에 루프가 완성됨
        call_ai()

    # ─── 경우 B: AI가 그냥 말로 답한 경우 ───
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"🎬 Agent: {message.content}")


def call_ai():
    """AI한테 messages + TOOLS 넘기고 응답 받기"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,  # ★ 사용 가능한 함수 설명서도 같이 넘김
    )
    process_ai_response(response.choices[0].message)


# =============================================================
# [4] 실행! - 사용자와 대화 시작
# =============================================================

print("=" * 50)
print("🎬 Movie Agent 시작! (종료: 'q' 또는 'quit')")
print("=" * 50)

while True:
    user_message = input("질문하세요...")
    if user_message == "quit" or user_message == "q":
        print("👋 Movie Agent 종료")
        break
    else:
        # 사용자 메시지를 messages에 저장
        messages.append({"role": "user", "content": user_message})
        print(f"👤 User: {user_message}")

        # AI한테 물어보기 시작!
        call_ai()


🎬 Movie Agent 시작! (종료: 'q' 또는 'quit')
👤 User: 공포영화에 대해 알려줘 
🔧 함수 호출: get_popular_movies({})
📋 인기 영화 목록 (ID / 제목):
   - ID 1523145: Your Heart Will Be Broken
   - ID 83533: Avatar: Fire and Ash
   - ID 1327819: Hoppers
   - ID 502356: The Super Mario Bros. Movie
   - ID 1290821: Shelter
   - ID 1171145: Crime 101
   - ID 840464: Greenland 2: Migration
   - ID 687163: Project Hail Mary
   - ID 1297842: GOAT
   - ID 1226863: The Super Mario Galaxy Movie
   - ID 1115544: Mike & Nick & Nick & Alice
   - ID 1641319: Sniper: No Nation
   - ID 1159831: The Bride!
   - ID 1265609: War Machine
   - ID 1470130: The Mortuary Assistant
   - ID 1084187: Pretty Lethal
   - ID 1268127: Humint
   - ID 1159559: Scream 7
   - ID 615: The Passion of the Christ
   - ID 662707: Starbright
✅ 결과 받음 (앞부분): [{"id": 1523145, "title": "Your Heart Will Be Broken", "overview": "High school student Polina is saved from bullying at...
🎬 Agent: 현재 인기 있는 공포 영화 목록입니다:
1. [The Mortuary Assistant](ID: 1470130) - 최근 검시학을 졸